# Predicting Mental Health Impact from Social Media Usage Patterns
**Dataset:** Social Media & Mental Health — Kaggle  
**Task:** Classification  
**Student:** Parvxz07

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)

print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load dataset (download from Kaggle and place in same folder)
# Dataset: https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health
df = pd.read_csv('smmh.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 3. Explore the Dataset

In [ ]:
print('Dataset Info:')
df.info()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
print('Basic statistics:')
df.describe()

## 4. Data Cleaning & Preprocessing

In [ ]:
# Rename columns for easier use
df.columns = [
    'timestamp', 'age', 'gender', 'relationship_status',
    'occupation', 'uses_social_media', 'platforms',
    'avg_time', 'no_purpose_use', 'distracted',
    'restless', 'easily_distracted', 'bothered_by_worries',
    'difficulty_concentrating', 'compare_self', 'feel_about_comparisons',
    'seek_validation', 'feel_depressed', 'interest_fluctuation',
    'sleep_issues'
]

# Drop timestamp and non-useful columns
df.drop(columns=['timestamp', 'uses_social_media', 'platforms'], inplace=True)

# Drop duplicates and missing values
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

print('Cleaned dataset shape:', df.shape)
df.head()

In [ ]:
# Create target variable: mental health impact (1 = affected, 0 = not affected)
# Based on avg daily usage time > 3 hours AND feeling depressed > 3
df['avg_time'] = df['avg_time'].astype(str)

time_map = {
    'Less than an Hour': 0.5,
    'Between 1 and 2 hours': 1.5,
    'Between 2 and 3 hours': 2.5,
    'Between 3 and 4 hours': 3.5,
    'Between 4 and 5 hours': 4.5,
    'More than 5 hours': 6.0
}
df['avg_time_num'] = df['avg_time'].map(time_map)
df.dropna(subset=['avg_time_num'], inplace=True)

# Target: high usage + feeling depressed
df['mental_health_impact'] = ((df['avg_time_num'] >= 3.5) & (df['feel_depressed'] >= 3)).astype(int)

print('Target variable distribution:')
print(df['mental_health_impact'].value_counts())
print(f"\nAffected: {df['mental_health_impact'].sum()} | Not affected: {(df['mental_health_impact']==0).sum()}")

In [ ]:
# Encode categorical columns
le = LabelEncoder()
for col in ['gender', 'relationship_status', 'occupation', 'avg_time']:
    df[col] = le.fit_transform(df[col].astype(str))

print('Encoding done. Data types:')
print(df.dtypes)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Figure 1: Target variable distribution
plt.figure(figsize=(6, 4))
labels = ['Not Affected', 'Affected']
counts = df['mental_health_impact'].value_counts().sort_index()
colors = ['#5DCAA5', '#D85A30']
plt.bar(labels, counts, color=colors, edgecolor='white', linewidth=0.5)
plt.title('Mental Health Impact Distribution', fontsize=14)
plt.ylabel('Number of Respondents')
plt.tight_layout()
plt.savefig('fig1_target_distribution.png', dpi=150)
plt.show()
print('Figure 1 saved.')

In [ ]:
# Figure 2: Age distribution
plt.figure(figsize=(7, 4))
plt.hist(df['age'], bins=20, color='#378ADD', edgecolor='white')
plt.title('Age Distribution of Respondents', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('fig2_age_distribution.png', dpi=150)
plt.show()
print('Figure 2 saved.')

In [ ]:
# Figure 3: Sleep issues vs mental health impact
plt.figure(figsize=(7, 4))
df.groupby('mental_health_impact')['sleep_issues'].mean().plot(
    kind='bar', color=['#5DCAA5', '#D85A30'], edgecolor='white'
)
plt.title('Average Sleep Issues by Mental Health Impact', fontsize=14)
plt.xlabel('Mental Health Impact (0=No, 1=Yes)')
plt.ylabel('Average Sleep Issues Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('fig3_sleep_vs_impact.png', dpi=150)
plt.show()
print('Figure 3 saved.')

In [ ]:
# Figure 4: Correlation heatmap
plt.figure(figsize=(10, 7))
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png', dpi=150)
plt.show()
print('Figure 4 saved.')

In [ ]:
# Figure 5: Seek validation vs depression
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x='seek_validation', y='feel_depressed',
                hue='mental_health_impact', palette={0:'#5DCAA5', 1:'#D85A30'}, alpha=0.7)
plt.title('Seek Validation vs Feel Depressed', fontsize=14)
plt.xlabel('Seek Validation Score')
plt.ylabel('Feel Depressed Score')
plt.legend(title='Affected', labels=['No', 'Yes'])
plt.tight_layout()
plt.savefig('fig5_validation_vs_depression.png', dpi=150)
plt.show()
print('Figure 5 saved.')

## 6. Prepare Features & Split Data

In [ ]:
# Define features and target
X = df.drop(columns=['mental_health_impact'])
y = df['mental_health_impact']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')

## 7. Train Machine Learning Models

In [ ]:
results = {}

# Model 1: Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr, zero_division=0),
    'predictions': y_pred_lr
}
print('Logistic Regression:')
print(classification_report(y_test, y_pred_lr, zero_division=0))

In [ ]:
# Model 2: Decision Tree
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
results['Decision Tree'] = {
    'accuracy': accuracy_score(y_test, y_pred_dt),
    'f1': f1_score(y_test, y_pred_dt, zero_division=0),
    'predictions': y_pred_dt
}
print('Decision Tree:')
print(classification_report(y_test, y_pred_dt, zero_division=0))

In [ ]:
# Model 3: Random Forest
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf, zero_division=0),
    'predictions': y_pred_rf
}
print('Random Forest:')
print(classification_report(y_test, y_pred_rf, zero_division=0))

In [ ]:
# Model 4: K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
results['KNN'] = {
    'accuracy': accuracy_score(y_test, y_pred_knn),
    'f1': f1_score(y_test, y_pred_knn, zero_division=0),
    'predictions': y_pred_knn
}
print('KNN:')
print(classification_report(y_test, y_pred_knn, zero_division=0))

## 8. Evaluate & Compare Models

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'F1 Score': [results[m]['f1'] for m in results]
})
summary = summary.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print('Model Comparison:')
print(summary.to_string(index=False))

In [ ]:
# Figure 6: Model comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summary))
width = 0.35
bars1 = ax.bar(x - width/2, summary['Accuracy'], width, label='Accuracy', color='#378ADD')
bars2 = ax.bar(x + width/2, summary['F1 Score'], width, label='F1 Score', color='#1D9E75')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy & F1 Score', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(summary['Model'], rotation=15)
ax.set_ylim(0, 1.1)
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('fig6_model_comparison.png', dpi=150)
plt.show()
print('Figure 6 saved.')

In [ ]:
# Figure 7: Confusion matrix for best model (Random Forest)
plt.figure(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Affected', 'Affected'],
            yticklabels=['Not Affected', 'Affected'])
plt.title('Confusion Matrix — Random Forest', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('fig7_confusion_matrix.png', dpi=150)
plt.show()
print('Figure 7 saved.')

In [ ]:
# Figure 8: Feature importance (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='#7F77DD', edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=14)
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig8_feature_importance.png', dpi=150)
plt.show()
print('Figure 8 saved.')

## 9. Conclusion

In [ ]:
best_model = summary.iloc[0]
print('=' * 50)
print('CONCLUSION')
print('=' * 50)
print(f"Best performing model: {best_model['Model']}")
print(f"Accuracy: {best_model['Accuracy']:.4f}")
print(f"F1 Score: {best_model['F1 Score']:.4f}")
print('\nKey Findings:')
print('- Social media usage time and depression scores are strong predictors')
print('- Users spending 3.5+ hours/day show higher mental health impact')
print('- Seeking validation online correlates with depression scores')
print('- Sleep issues are closely linked with mental health impact')
print('- Random Forest outperformed other models on this dataset')